# Fine-tuning com Correções Reais — Classificador de Soja

Este notebook pega o modelo já treinado (`soja_model_final.keras`) e o **ajusta** com as fotos
que você corrigiu no app (salvas no dataset `Guguinhaxd/soja-correction`).

**Por que isso resolve o domain shift:** o modelo foi treinado com fotos de laboratório (câmera do dataset).
As suas fotos de celular parecem diferentes. As correções são exemplos do **seu** setup real — fine-tuning
com elas ensina o modelo a reconhecer os grãos do jeito que VOCÊ fotografa.

### Cuidado central: *catastrophic forgetting*
Se treinarmos SÓ com as poucas correções, o modelo esquece tudo que sabia. A solução aqui é
**misturar correções + dataset original** (replay) — metade de cada lote vem de cada fonte.

### Pré-requisitos
1. Ter rodado `train.ipynb` antes → `soja_model_final.keras` no Drive
2. Ter pelo menos ~8-10 correções por classe no dataset (quanto mais, melhor)
3. GPU ligada: `Ambiente de execução → Alterar tipo → T4 GPU`

In [ ]:
# Célula 1 — GPU, imports e login no Hugging Face
!pip install -q huggingface_hub

import os, json, glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from huggingface_hub import snapshot_download, HfApi

print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
assert len(gpus) > 0, 'Sem GPU! Ambiente de execução → Alterar tipo → T4'

# Cole seu token do HF (o mesmo SOJA_CORRECTIONS, precisa de permissão de leitura).
# Pegue em: https://huggingface.co/settings/tokens
from getpass import getpass
HF_TOKEN = getpass('Cole seu token do Hugging Face: ')

In [ ]:
# Célula 2 — Montar Drive, caminhos e classes
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT     = '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)'
TRAIN_DIR        = f'{DATASET_ROOT}/train'
VALID_DIR        = f'{DATASET_ROOT}/valid'
TEST_DIR         = f'{DATASET_ROOT}/test'
SAVE_DIR         = '/content/drive/MyDrive'
BASE_MODEL_PATH  = f'{SAVE_DIR}/soja_model_final.keras'

CORRECTIONS_REPO = 'Guguinhaxd/soja-correction'

# Mesma ordem do train.ipynb — NÃO mudar (índice = saída do modelo)
CLASS_NAMES = [
    'Broken soybeans',
    'Immature soybeans',
    'Intact soybeans',
    'Skin-damaged soybeans',
    'Spotted soybeans',
]
SHORT = {
    'Broken soybeans':       'broken',
    'Immature soybeans':     'immature',
    'Intact soybeans':       'intact',
    'Skin-damaged soybeans': 'skin-damaged',
    'Spotted soybeans':      'spotted',
}
NUM_CLASSES = len(CLASS_NAMES)

assert os.path.isfile(BASE_MODEL_PATH), f'Modelo base não encontrado: {BASE_MODEL_PATH}'
assert os.path.isdir(TRAIN_DIR), f'Dataset original não encontrado: {TRAIN_DIR}'
print('Modelo base e dataset original OK.')

In [ ]:
# Célula 3 — Baixar as correções do Hugging Face e contar por classe
CORR_DIR = snapshot_download(
    repo_id=CORRECTIONS_REPO,
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir='/content/correcoes',
)
print(f'Correções baixadas em: {CORR_DIR}\n')

# As fotos ficam em subpastas com o nome EXATO da classe (ver app.py: path = f'{correct_class}/...').
corr_counts = {}
for cls in CLASS_NAMES:
    files = glob.glob(os.path.join(CORR_DIR, cls, '*.jpg')) + \
            glob.glob(os.path.join(CORR_DIR, cls, '*.jpeg')) + \
            glob.glob(os.path.join(CORR_DIR, cls, '*.png'))
    corr_counts[cls] = files
    print(f'{SHORT[cls]:>14}: {len(files):>3} correções')

total_corr = sum(len(v) for v in corr_counts.values())
print(f'\nTotal de correções: {total_corr}')
assert total_corr >= 5, 'Poucas correções! Colete mais fotos pelo app antes de re-treinar.'
if total_corr < NUM_CLASSES * 8:
    print('\n⚠️  Recomendado ter ~8+ por classe para um fine-tuning sólido. Dá pra rodar mesmo assim.')

In [ ]:
# Célula 4 — Montar lista (caminho, rótulo) das correções com índice CORRETO
# Construímos manualmente (em vez de image_dataset_from_directory) para garantir que o
# índice de cada classe bate com o do modelo, mesmo que alguma classe não tenha correção.
corr_paths, corr_labels = [], []
for idx, cls in enumerate(CLASS_NAMES):
    for fp in corr_counts[cls]:
        corr_paths.append(fp)
        corr_labels.append(idx)

corr_paths  = np.array(corr_paths)
corr_labels = np.array(corr_labels)
print(f'{len(corr_paths)} imagens de correção prontas.')

# Separar ~20% das correções para VALIDAÇÃO (ver se o fine-tuning realmente ajudou no seu setup)
rng = np.random.default_rng(42)
perm = rng.permutation(len(corr_paths))
n_val_corr = max(NUM_CLASSES, int(len(corr_paths) * 0.2))
val_idx, train_idx = perm[:n_val_corr], perm[n_val_corr:]

ct_paths,  ct_labels  = corr_paths[train_idx], corr_labels[train_idx]
cv_paths,  cv_labels  = corr_paths[val_idx],   corr_labels[val_idx]
print(f'Correções p/ treino: {len(ct_paths)} | p/ validação: {len(cv_paths)}')

In [ ]:
# Célula 5 — Pipelines tf.data
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def paths_to_ds(paths, labels, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=42, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE)

# Dataset ORIGINAL (replay anti-esquecimento)
orig_train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
    batch_size=None, label_mode='categorical', shuffle=True, seed=42,
)
orig_train = orig_train.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)

# Correções (elemento a elemento, com repeat porque são poucas)
corr_train = paths_to_ds(ct_paths, ct_labels, shuffle=True)

# Augmentation leve (vale pros dois)
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.08),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

# MISTURA 50/50: metade dos exemplos vem das suas correções, metade do dataset original.
# .repeat() nas correções para não acabarem antes; sample_from_datasets faz o balanço.
mixed = tf.data.Dataset.sample_from_datasets(
    [corr_train.repeat(), orig_train.repeat()],
    weights=[0.5, 0.5], seed=42,
)
mixed = (mixed
         .batch(BATCH_SIZE)
         .map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
         .prefetch(AUTOTUNE))

# Validação = SUAS correções separadas (mede melhora no seu domínio real)
corr_val = paths_to_ds(cv_paths, cv_labels, shuffle=False).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Quantos passos por época? ~2x o original ÷ batch já dá bastante exposição às correções.
STEPS_PER_EPOCH = max(50, (len(ct_paths) * 4) // BATCH_SIZE)
print(f'Steps por época: {STEPS_PER_EPOCH}')

In [ ]:
# Célula 6 — Carregar o modelo treinado e preparar fine-tuning
model = keras.models.load_model(BASE_MODEL_PATH)
print('Modelo carregado.')

# Acurácia ANTES do fine-tuning, nas suas correções (baseline pra comparar depois)
base_loss, base_acc = model.evaluate(corr_val, verbose=0)
print(f'ANTES — acurácia nas suas fotos: {base_acc:.1%}')

# Descongelar só as últimas 30 camadas da base EfficientNet, mantendo BatchNorm congelada
base_model = None
for layer in model.layers:
    if isinstance(layer, keras.Model):  # a base EfficientNet vem aninhada como submodelo
        base_model = layer
        break

if base_model is not None:
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    trainable = sum(1 for l in base_model.layers if l.trainable)
    print(f'Camadas treináveis na base: {trainable}/{len(base_model.layers)}')
else:
    print('Base aninhada não encontrada — treinando só a cabeça (ainda funciona).')

In [ ]:
# Célula 7 — Treinar (LR bem baixo, poucas épocas)
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # LR baixo: ajusta sem destruir o que já sabe
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=4, restore_best_weights=True,
        monitor='val_accuracy', verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{SAVE_DIR}/soja_finetuned_best.keras',
        save_best_only=True, monitor='val_accuracy', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        patience=2, factor=0.5, monitor='val_loss',
        min_lr=1e-7, verbose=1),
]

print('=== Fine-tuning com correções reais ===')
history = model.fit(
    mixed,
    validation_data=corr_val,
    epochs=12,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks,
)

In [ ]:
# Célula 8 — Comparar ANTES vs DEPOIS
new_loss, new_acc = model.evaluate(corr_val, verbose=0)
print(f'ANTES  — acurácia nas suas fotos: {base_acc:.1%}')
print(f'DEPOIS — acurácia nas suas fotos: {new_acc:.1%}')
print(f'Ganho: {(new_acc - base_acc)*100:+.1f} pontos percentuais')

# Sanidade: confere que NÃO esqueceu o dataset original
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False,
).map(lambda x, y: (tf.cast(x, tf.float32), y)).prefetch(AUTOTUNE)
_, test_acc = model.evaluate(test_ds, verbose=0)
print(f'\nAcurácia no TESTE original (não pode despencar): {test_acc:.1%}')

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Treino (misto)')
plt.plot(history.history['val_accuracy'], label='Validação (suas fotos)')
plt.title('Acurácia'); plt.xlabel('Época'); plt.legend(); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Loss'); plt.xlabel('Época'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Célula 9 — Salvar modelo final ajustado
# Decisão: só sobrescreve o modelo de produção se DEPOIS >= ANTES no seu domínio
#          e o teste original não caiu de forma absurda.
FINETUNED_PATH = f'{SAVE_DIR}/soja_finetuned_final.keras'
model.save(FINETUNED_PATH)
print(f'Modelo ajustado salvo em: {FINETUNED_PATH}')

if new_acc >= base_acc and test_acc >= 0.70:
    model.save(f'{SAVE_DIR}/soja_model_final.keras')
    print('✅ Melhorou (ou manteve) nas suas fotos e segurou o teste original.')
    print('   Sobrescrevi soja_model_final.keras — pronto para o Space.')
else:
    print('⚠️  Resultado não melhorou claramente. Mantive o soja_model_final.keras antigo.')
    print('   Colete mais correções e rode de novo, ou use o soja_finetuned_final.keras manualmente.')

In [ ]:
# Célula 10 (OPCIONAL) — Subir o modelo ajustado direto pro HF Space
# Evita o download/git manual. Usa o mesmo token (precisa permissão de escrita no Space).
SPACE_REPO = 'Guguinhaxd/soja-inspection'

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=f'{SAVE_DIR}/soja_model_final.keras',
    path_in_repo='soja_model_final.keras',
    repo_id=SPACE_REPO,
    repo_type='space',
    commit_message='fine-tune: modelo ajustado com correções reais',
)
print('✅ Modelo enviado pro Space. Ele vai rebuildar sozinho em ~2 min.')